## 1. Setup & Data Loading

In [1]:
from pyspark.sql import SparkSession

# Initialise Spark session
spark = (
    SparkSession.builder.appName("Rideshare_Analysis")
    .config("spark.sql.repl.eagerEval.enabled", True) 
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "12g")
    .config("spark.driver.maxResultSize", "2g")
    .config("spark.sql.shuffle.partitions", "400")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .getOrCreate()
)


your 131072x1 screen size is bogus. expect trouble
24/08/26 23:17:26 WARN Utils: Your hostname, LAPTOP-354CCOC2 resolves to a loopback address: 127.0.1.1; using 172.21.14.166 instead (on interface eth0)
24/08/26 23:17:26 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/08/26 23:17:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
# When session restarts, continue progress from here
from pyspark.sql.types import *
from pyspark.ml.linalg import VectorUDT

# Define the schema
schema = StructType([
    StructField("hvfhs_license_num", StringType(), True),
    StructField("PULocationID", IntegerType(), True),
    StructField("trip_miles", DoubleType(), True),
    StructField("license_vec", VectorUDT(), True),
    StructField("day_of_week", IntegerType(), True),
    StructField("hour_of_day", IntegerType(), True),
    StructField("month", IntegerType(), True),
    StructField("day_vec", VectorUDT(), True),
    StructField("hour_vec", VectorUDT(), True),
    StructField("PULocation_vec", VectorUDT(), True),
    StructField("trip_miles_standardised", DoubleType(), True),
    StructField("trip_time_standardised", DoubleType(), True),
    StructField("earnings_per_hour", DoubleType(), True),
    StructField("earnings", DoubleType(), True),
    StructField("feelslike", DoubleType(), True),
    StructField("feelslike_standardised", DoubleType(), True),
    StructField("precip_standardised", DoubleType(), True),
    StructField("preciptype_vec", VectorUDT(), True),
    StructField("is_public_holiday", IntegerType(), True)
])

# Load the Parquet file with the defined schema
final_df = spark.read.schema(schema).parquet("../data/curated/chunk_*.parquet")


## 2. Feature Assembly

Features: encoded license, standardised trip miles, day/hour/location encodings, weather (feels-like temp, precipitation amount + type), public holiday flag.

In [4]:
# Prepare Data for Linear Regression
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression

# Feature columns (excludes the target variable)
feature_columns = [
    'license_vec', 'trip_miles_standardised','day_vec', 'hour_vec', 'PULocation_vec', 
    'feelslike_standardised', 'precip_standardised', 'preciptype_vec', 'is_public_holiday'
]

# Assemble the features into a feature vector
assembler = VectorAssembler(inputCols=feature_columns, outputCol='features')
assembled_df = assembler.transform(final_df)
final_data = assembled_df.select("features", "earnings")

## 3. Train/Test Split

Temporal holdout strategy: train on May–October 2023 (months 5–10), test on November 2023 (month 11). This avoids data leakage and tests how well the model generalises to unseen future data.

In [5]:
# Split into train and test sets
from pyspark.sql.functions import col

# Filter the DataFrame to create training and test sets
training_data = final_data.filter(col('month').between(5, 10))  # May to October
test_data = final_data.filter(col('month') == 11)  # November

# Inspect size of sets
print("Training Set: ", training_data.count())
print("Test Set: ", test_data.count())

# Preview
training_data.limit(7)


Training Set:  90139613


Test Set:  14951054


features,earnings
"(295,[1,25,183,29...",10.27
"(295,[1,25,167,29...",7.75
"(295,[0,1,25,105,...",10.57
"(295,[0,1,25,225,...",24.27
"(295,[0,1,25,42,2...",7.12
"(295,[0,1,25,86,2...",32.93
"(295,[0,1,25,101,...",5.38


## 4. Linear Regression

In [6]:
# Coefficients, Predictions/Result

# Initialise the Linear Regression model
lr = LinearRegression(featuresCol='features', 
                      labelCol='earnings', 
                     )

#Fit model on training data
lm = lr.fit(training_data)

# Output the coefficients and intercept
print("Coefficients: ", lm.coefficients)
print("Intercept: ", lm.intercept)

# Generate predictions for test data
predictions = lm.transform(test_data)

# Display some of the predictions
predictions.select('features', 'earnings', 'prediction').limit(7)

24/08/26 23:17:46 WARN Instrumentation: [b4821329] regParam is zero, which might cause numerical instability and overfitting.
24/08/26 23:18:37 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
24/08/26 23:19:20 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK


Coefficients:  [0.45167688602564915,11.672547719377317,0.3320421623595751,0.9534962817150325,0.9842843156941306,0.015261231694377627,0.6645451431634009,0.43351969686432446,3.6968232392209908,2.665063285878961,4.58945623860574,1.6467605034220199,1.8557041029008463,1.7312586175203943,4.749382710931235,4.760742403662563,1.7444983871041895,4.354493297003127,2.9871453816820788,2.416674697252472,3.7269667067843133,3.237796581867322,2.520480549326586,2.906387601917116,2.51653499998218,1.2042828851327743,0.4641637994970944,0.5059575546900371,0.24954519045927157,-0.7602514144408088,-0.4327380924542715,3.431995653665997,0.6954669897133023,4.531376051007789,3.4159851443041167,6.249211004786559,5.855196464446122,5.151537227375548,5.569283072373179,5.603986731243428,4.9380580771524745,4.769597711937112,5.096273682037895,3.9032920648587712,1.642399005646015,4.631800262777827,4.7914564187441115,1.8318523832894944,5.451817188654133,4.013095831223774,3.8810095073874455,1.7279549849099074,5.875317592267

features,earnings,prediction
"(295,[0,1,4,18,14...",10.79,10.692969753356012
"(295,[0,1,4,18,95...",5.39,8.172654667837147
"(295,[0,1,4,18,17...",12.64,12.046363362267646
"(295,[0,1,4,18,79...",41.96,40.93194258443201
"(295,[1,4,18,79,2...",5.47,6.621285279801881
"(295,[0,1,4,18,45...",11.51,13.91415680217635
"(295,[0,1,4,18,42...",10.64,13.891099905833762


In [7]:
# Evaluate RMSE on test set
from pyspark.ml.evaluation import RegressionEvaluator

#Linear Regression
evaluator = RegressionEvaluator(
    labelCol='earnings', predictionCol='prediction', metricName='rmse'
)

rmse = evaluator.evaluate(predictions)
print("Root Mean Squared Error (RMSE) on test data = %g" % rmse)

Root Mean Squared Error (RMSE) on test data = 4.88163


In [8]:
from pyspark.ml.evaluation import RegressionEvaluator

# Linear Regression
evaluator = RegressionEvaluator(
    labelCol='earnings', predictionCol='prediction', metricName='r2'
)

r2 = evaluator.evaluate(predictions)
print("R-squared (R2) on test data = %g" % r2)


R-squared (R2) on test data = 0.849963


In [9]:
# Save linear reg data
linear_predictions = predictions.select('earnings', 'prediction')
linear_predictions.write.mode('overwrite').parquet(f"../data/results/linear_pred.parquet")

# Save actual data
actual_values = test_data.select('earnings')
actual_values.write.mode('overwrite').parquet(f"../data/results/actual_values.parquet")

## 5. Regularised Linear Regression (Lasso)

Lasso regression (`elasticNetParam=1.0`) adds L1 regularisation, which shrinks less informative feature coefficients toward zero. `regParam=1` was selected as a moderate regularisation strength; higher values increase sparsity.

In [10]:
# Prepare Data for RandomForest Regression
from pyspark.ml.feature import VectorAssembler

# Feature columns (excludes the target variable)
feature_columns = [
    'license_vec', 'trip_miles_standardised','day_vec', 'hour_vec', 'PULocation_vec', 
    'feelslike_standardised', 'precip_standardised', 'preciptype_vec', 'is_public_holiday'
]

# Assemble the features into a feature vector
assembler = VectorAssembler(inputCols=feature_columns, outputCol='features')
assembled_df = assembler.transform(final_df)
final_data = assembled_df.select("features", "earnings")

In [11]:
# Split into train and test sets
from pyspark.sql.functions import col

# Filter the DataFrame to create training and test sets
training_data = final_data.filter(col('month').between(5, 10))  # May to October
test_data = final_data.filter(col('month') == 11)  # November

# Inspect size of sets
print("Training Set: ", training_data.count())
print("Test Set: ", test_data.count())

# Preview
training_data.limit(7)

Training Set:  90139613


Test Set:  14951054


features,earnings
"(295,[1,25,183,29...",10.27
"(295,[1,25,167,29...",7.75
"(295,[0,1,25,105,...",10.57
"(295,[0,1,25,225,...",24.27
"(295,[0,1,25,42,2...",7.12
"(295,[0,1,25,86,2...",32.93
"(295,[0,1,25,101,...",5.38


In [12]:
# Coefficients, Predictions/Result

# lasso_reg_param controls the amount of regularization (L1 regularization)
lasso_reg_param = 1

# Initialise the Linear Regression model
lr = LinearRegression(featuresCol='features', 
                      labelCol='earnings', 
                      elasticNetParam=1.0,  # Setting this to 1.0 applies Lasso regularization
                      regParam=lasso_reg_param  # The regularization strength, higher means more regularization
                     )

#Fit model on training data
lm = lr.fit(training_data)

# Output the coefficients and intercept
print("Coefficients: ", lm.coefficients)
print("Intercept: ", lm.intercept)

# Generate predictions for test data
predictions = lm.transform(test_data)

# Display some of the predictions
predictions.select('features', 'earnings', 'prediction').limit(7)

ERROR:root:KeyboardInterrupt while sending command.               (0 + 16) / 20]
Traceback (most recent call last):
  File "/home/ryan/.local/lib/python3.10/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/home/ryan/.local/lib/python3.10/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/usr/lib/python3.10/socket.py", line 705, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt


KeyboardInterrupt: 

In [ ]:
# Save regularised linear reg data
reg_linear_predictions = predictions.select('earnings', 'prediction')
reg_linear_predictions.write.mode('overwrite').parquet(f"../data/results/regularised_linear_pred.parquet")

## 6. Lasso Evaluation

In [ ]:
# Evaluate RMSE on test set
from pyspark.ml.evaluation import RegressionEvaluator

evaluator = RegressionEvaluator(labelCol="earnings", predictionCol="prediction", metricName="rmse")
rmse_rf = evaluator.evaluate(predictions)

print(f"Root Mean Squared Error (RMSE) on test data = {rmse_rf}")



Root Mean Squared Error (RMSE) on test data = 5.422052550162519


In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator

evaluator = RegressionEvaluator(
    labelCol='earnings', predictionCol='prediction', metricName='r2'
)

r2 = evaluator.evaluate(predictions)
print("R-squared (R2) on test data = %g" % r2)


R-squared (R2) on test data = 0.814904


## 7. Results Visualisation

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")

# Read in results
linear_predictions_df = pd.read_parquet("../data/results/linear_pred.parquet")
regularised_linear_predictions_df = pd.read_parquet("../data/results/regularised_linear_pred.parquet")
actual_values_df = pd.read_parquet("../data/results/actual_values.parquet")

# Scatter plot
plt.figure(figsize=(12, 6))

# Correcting the column access
plt.scatter(actual_values_df['earnearningsings'], linear_predictions_df['prediction'], label='Linear Regression', color='green', alpha=0.6)
plt.scatter(actual_values_df[''], regularised_linear_predictions_df['prediction'], label='Lasso Linear Regression', color='red', alpha=0.6)

plt.xlabel('Actual Earnings')
plt.ylabel('Predicted Earnings')
plt.title('Scatter Plot: Predicted vs Actual Values')
plt.legend()
plt.grid(True)

# Save the plot to a plots folder
plt.savefig('../plots/Predictions_Comparison.png')
plt.show()


: 